# Volet 1 — Exécution du pipeline de nettoyage
### Notebook d'exécution — du fichier brut au corpus prêt pour l'apprentissage

Ce notebook **lance réellement** les étapes du pipeline (`step1` à `step3`) et
affiche, entre chaque étape, ce qui a changé. Il appelle les modules du projet
plutôt que de recopier leur code : les chiffres obtenus ici sont donc exactement
ceux des scripts en ligne de commande, et du rapport.

Ordre : diagnostic (notebook 01) → **nettoyage (ce notebook)** → apprentissage.

> ⚠️ Les cellules qui affichent des réclamations montrent du texte client réel.
> Ne pas capturer ces sorties-là dans le rapport sans masquer les noms.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # racine nlp-pipeline/

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from src import cleaning_rules as rules

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

GREEN_DARK  = "#1B5E3A"
GREEN_MED   = "#2E7D4F"
GREEN_LIGHT = "#7FB89A"
GREY_DARK   = "#3A3A3A"
GREY_MED    = "#6B6B6B"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": GREY_MED,
    "axes.labelcolor": GREY_DARK,
    "text.color": GREY_DARK,
    "xtick.color": GREY_DARK,
    "ytick.color": GREY_DARK,
    "font.size": 11,
    "axes.titleweight": "bold",
    "axes.titlecolor": GREY_DARK,
})

def savefig(fig, name):
    path = config.FIGURES / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    print(f"Figure enregistrée : {path}")

print("Paramètres de cette exécution")
print(f"  Échantillon de travail      : {config.CORPUS_SAMPLE_FRACTION:.0%} du corpus exploitable")
print(f"  Plancher par catégorie      : {config.MIN_SAMPLES_PER_CLASS} exemples")
print(f"  Longueur minimale de texte  : {config.MIN_DESCRIPTION_CHARS} caractères")
print(f"  Modèle LLM                  : {config.LLM_MODEL}")
print(f"  Cibles                      : {list(config.TARGETS)}")

## 1. Préparation du corpus (`step1`)
*[Chapitre 4.1 — filtrage et suppression des colonnes]*

Cette étape enlève les actes automatiques, supprime les colonnes demandées par
le service, assemble les deux cibles, puis tire l'échantillon de travail.

**Les deux colonnes cibles sont conservées**, sous leur nom de travail :

| Colonne d'origine | Nom dans le corpus nettoyé | Rôle |
|---|---|---|
| `LIBELLE` | `REQUEST_CATEGORY` | catégorie à prédire |
| `NIVEAU_TRAITEMENT` | `ROUTING_LEVEL` | niveau FO / MO / BO à prédire |

Elles ne sont jamais données au modèle en entrée — uniquement comme étiquettes.

In [ ]:
from pipeline import step1_prepare

step1_prepare.main()

In [ ]:
prepared = pd.read_parquet(config.PREPARED_PARQUET)

print(f"Corpus préparé : {len(prepared):,} lignes x {prepared.shape[1]} colonnes\n")
print("Colonnes conservées :")
for column in prepared.columns:
    print(f"  - {column}")

print(f"\nColonnes supprimées : {len(step1_prepare.DROPPED_COLUMNS)}")
print("Aucune donnée personnelle ne subsiste : ni nom, ni date de naissance,")
print("ni adresse, ni e-mail, ni identifiant client.")

### L'entonnoir de filtrage
*[Tableau 1 du rapport]*

In [ ]:
stats = json.loads((config.METRICS / "step1_prepare.json").read_text(encoding="utf-8"))

etapes = [
    ("Export brut", stats["rows_raw"]),
    ("− doublons exacts", -stats["exact_duplicate_rows_dropped"]),
    ("− pièces jointes corrompues", -stats["office_artefacts_dropped"]),
    ("− actes automatiques", -stats["automated_acts_dropped"]),
    ("− descriptions trop courtes", -stats["short_descriptions_dropped"]),
    ("− lignes sans étiquette", -stats.get("unlabelled_rows_dropped", 0)),
]

courant = stats["rows_raw"]
niveaux, labels = [], []
for label, delta in etapes:
    courant = delta if label == "Export brut" else courant + delta
    niveaux.append(courant)
    labels.append(label)

usable = stats.get("rows_before_sampling", stats["rows_kept"])
niveaux.append(usable)
labels.append("= corpus exploitable")
niveaux.append(stats["rows_kept"])
labels.append(f"échantillon {config.CORPUS_SAMPLE_FRACTION:.0%}")

fig, ax = plt.subplots(figsize=(10, 5))
couleurs = [GREY_DARK] + [GREEN_LIGHT] * (len(niveaux) - 3) + [GREEN_MED, GREEN_DARK]
bars = ax.bar(labels, niveaux, color=couleurs)
for bar, valeur in zip(bars, niveaux):
    ax.text(bar.get_x() + bar.get_width() / 2, valeur, f"{valeur:,}",
            ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Lignes restantes")
ax.set_title("Du fichier source au corpus de travail")
ax.set_ylim(0, max(niveaux) * 1.15)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
savefig(fig, "pipeline_entonnoir")
plt.show()

part = 100 * stats["rows_kept"] / stats["rows_raw"]
print(f"\nCorpus de travail : {stats['rows_kept']:,} réclamations ({part:.1f}% de l'export)")
if "rows_before_sampling" in stats:
    reel = 100 * stats["rows_kept"] / stats["rows_before_sampling"]
    print(f"Soit {reel:.1f}% du corpus exploitable "
          f"(visé : {config.CORPUS_SAMPLE_FRACTION:.0%} ; l'écart vient du plancher "
          f"de {config.MIN_SAMPLES_PER_CLASS} exemples par catégorie)")

### L'échantillon a-t-il conservé toutes les catégories ?

C'est le point sensible d'un tirage : réduire le corpus ne doit pas faire
disparaître des catégories entières. Le plancher par catégorie est là pour ça.

In [ ]:
counts = prepared["REQUEST_CATEGORY"].value_counts()

fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * min(len(counts), 25))))
top = counts.head(25).sort_values()
couleurs = [GREY_MED if v <= config.MIN_SAMPLES_PER_CLASS else GREEN_MED for v in top.values]
ax.barh(top.index, top.values, color=couleurs)
ax.axvline(config.MIN_SAMPLES_PER_CLASS, color=GREY_DARK, linestyle="--", linewidth=0.9)
ax.text(config.MIN_SAMPLES_PER_CLASS, -0.6, f" plancher = {config.MIN_SAMPLES_PER_CLASS}",
        fontsize=8, color=GREY_DARK)
ax.set_xlabel("Réclamations dans l'échantillon")
ax.set_title(f"Catégories retenues ({len(counts)} au total)")
plt.tight_layout()
savefig(fig, "pipeline_categories_echantillon")
plt.show()

print(f"Catégories présentes dans l'échantillon : {len(counts)}")
print(f"Catégories au plancher exactement       : {(counts <= config.MIN_SAMPLES_PER_CLASS).sum()}")
print(f"\nNiveaux de traitement :")
print(prepared["ROUTING_LEVEL"].value_counts().to_string())

## 2. Nettoyage par règles (`step2`)
*[Chapitre 4.2 — ce qu'une expression régulière sait faire]*

Retire des descriptions : salutations, formules de politesse et le bloc
signature qui les suit, les libellés de contact (adresse, téléphone, e-mail) et
les identifiants (numéro client, CIN, référence de dossier).

In [ ]:
from pipeline import step2_clean_rules

step2_clean_rules.main()

In [ ]:
cleaned = pd.read_parquet(config.RULES_PARQUET)

avant = cleaned["DESCRIPTION"].fillna("").str.len()
apres = cleaned["DESCRIPTION_RULES"].fillna("").str.len()

fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(0, 400, 40)
ax.hist(avant.clip(upper=400), bins=bins, alpha=0.75, color=GREY_MED, label="Avant nettoyage")
ax.hist(apres.clip(upper=400), bins=bins, alpha=0.75, color=GREEN_DARK, label="Après nettoyage")
ax.set_xlabel("Longueur de la description (caractères)")
ax.set_ylabel("Nombre de réclamations")
ax.set_title("Effet du nettoyage par règles sur la longueur du texte")
ax.legend(frameon=False)
plt.tight_layout()
savefig(fig, "pipeline_longueur_regles")
plt.show()

print(f"Longueur moyenne avant : {avant.mean():.0f} caractères")
print(f"Longueur moyenne après : {apres.mean():.0f} caractères")
print(f"Réduction              : {100 * (1 - apres.mean() / avant.mean()):.1f}%")
print(f"\nDescriptions distinctes : {cleaned['DESCRIPTION'].nunique():,} "
      f"-> {cleaned['DESCRIPTION_RULES'].nunique():,}")

### Exemples avant / après
⚠️ Texte client réel — masquer les noms avant toute capture d'écran.

In [ ]:
exemples = cleaned[cleaned["DESCRIPTION"].str.len() > 120].head(4)

for _, ligne in exemples.iterrows():
    print("AVANT :", ligne["DESCRIPTION"][:260])
    print("APRÈS :", ligne["DESCRIPTION_RULES"][:260])
    print("-" * 100)

## 3. Nettoyage sémantique par LLM (`step3`)
*[Chapitre 4.2 — ce qu'une expression régulière ne sait pas faire]*

Ce qu'aucune règle ne peut faire : reconnaître un nom propre écrit sans libellé,
et réécrire une réclamation noyée dans trois paragraphes de politesse en une
phrase exploitable. Le modèle tourne **en local** via Ollama : les réclamations
sont des données client et ne doivent pas quitter la machine.

**Prérequis** — dans un terminal séparé :

```
ollama pull qwen3:8b
ollama serve
```

La cellule suivante vérifie qu'Ollama répond avant de lancer l'étape, qui est
la plus longue du pipeline.

In [ ]:
ollama_ok = False
try:
    from langchain_ollama import ChatOllama
    ChatOllama(model=config.LLM_MODEL, temperature=0).invoke("Réponds OK.")
    ollama_ok = True
    print(f"Ollama répond, modèle '{config.LLM_MODEL}' disponible.")
except Exception as error:
    print("Ollama n'est pas joignable :")
    print(f"  {type(error).__name__}: {error}")
    print("\nLancez 'ollama serve' puis 'ollama pull qwen3:8b' et réexécutez cette cellule.")
    print("Sans lui, l'étape 3 se contente de recopier le texte nettoyé par règles :")
    print("la variante 'rules+llm' serait alors identique à 'rules'.")

In [ ]:
if ollama_ok:
    from pipeline import step3_clean_llm
    step3_clean_llm.main()
else:
    print("Étape ignorée : Ollama indisponible.")

In [ ]:
def llm_a_jour() -> bool:
    """La sortie de l'étape 3 correspond-elle au corpus courant ?

    Un 03_llm_cleaned.parquet peut rester d'une exécution précédente, faite sur
    un échantillon d'une autre taille. S'il est plus ancien que la sortie de
    l'étape 2, il décrit un autre corpus et ne doit pas être lu.
    """
    if not config.LLM_PARQUET.exists():
        return False
    return config.LLM_PARQUET.stat().st_mtime >= config.RULES_PARQUET.stat().st_mtime


if llm_a_jour():
    llm = pd.read_parquet(config.LLM_PARQUET)
    stats3 = json.loads((config.METRICS / "step3_llm.json").read_text(encoding="utf-8"))

    passes = [stats3[cle] for cle in ("conclusion_pass", "description_pass") if cle in stats3]
    for passe in passes:
        total = passe["rows"]
        distincts = passe["unique_values"]
        fig, ax = plt.subplots(figsize=(7, 4.5))
        bars = ax.bar(["Lignes à traiter", "Appels réellement faits"],
                      [total, distincts], color=[GREY_MED, GREEN_DARK], width=0.55)
        for bar, valeur in zip(bars, [total, distincts]):
            ax.text(bar.get_x() + bar.get_width() / 2, valeur, f"{valeur:,}",
                    ha="center", va="bottom", fontsize=11, fontweight="bold")
        economie = 100 * (1 - distincts / total) if total else 0
        ax.set_title(f"{passe['column']} — déduplication avant appel\n"
                     f"{economie:.0f}% d'appels évités")
        ax.set_ylim(0, total * 1.2)
        plt.tight_layout()
        savefig(fig, f"pipeline_llm_dedup_{passe['column'].lower()}")
        plt.show()
else:
    print("Pas encore de sortie à jour pour l'étape 3 sur ce corpus.")

### Ce que le LLM a réécrit
⚠️ Texte client réel.

In [ ]:
if llm_a_jour() and "DESCRIPTION_LLM" in llm.columns:
    modifies = llm[llm["DESCRIPTION_RULES"] != llm["DESCRIPTION_LLM"]]
    print(f"Descriptions réécrites par le LLM : {len(modifies):,} sur {len(llm):,}\n")
    for _, ligne in modifies.head(4).iterrows():
        print("RÈGLES :", str(ligne["DESCRIPTION_RULES"])[:240])
        print("LLM    :", str(ligne["DESCRIPTION_LLM"])[:240])
        print("-" * 100)
else:
    print("Étape 3 non exécutée : rien à comparer.")

## Résumé chiffré — à reporter dans le rapport et les diapositives

In [ ]:
final = pd.read_parquet(config.LLM_PARQUET if llm_a_jour() else config.RULES_PARQUET)

resume = {
    "Lignes dans l'export brut": f"{stats['rows_raw']:,}",
    "Actes automatiques écartés": f"{stats['automated_acts_dropped']:,}",
    "Colonnes supprimées": f"{stats.get('columns_dropped', 0)} sur {stats['columns_raw']}",
    "Corpus exploitable": f"{stats.get('rows_before_sampling', stats['rows_kept']):,}",
    f"Échantillon de travail ({config.CORPUS_SAMPLE_FRACTION:.0%})": f"{stats['rows_kept']:,}",
    "Corpus final après nettoyage": f"{len(final):,}",
    "Catégories (REQUEST_CATEGORY)": final["REQUEST_CATEGORY"].nunique(),
    "Niveaux de traitement": final["ROUTING_LEVEL"].nunique(),
    "Réduction moyenne du texte": f"{100 * (1 - apres.mean() / avant.mean()):.1f}%",
}

for label, valeur in resume.items():
    print(f"{label:<48} {valeur}")

print(f"\nFigures enregistrées dans : {config.FIGURES}")
print("\nÉtape suivante : python pipeline/step4_train.py")